# Batch nucleus segmentation for INDI production

## Experimental metadata extraction

### Metadata from epxerimental design

In [6]:
# load in metadata files
import pandas as pd

automated_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_automated_plates_combined.csv")
manual_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_manual_plates_combined.csv")
plate_id = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv")

### Metadata from Opera Phenix

In [7]:
# Define relevant dictionaries for metatdata

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "Alexa 568": "red",
    "Alexa 647": "magenta"
}

# Define pseudocolor mapping rules for matplotlib - helpful to generate images on the fly
mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

In [8]:
# 0c2035b2-6f9d-45b0-919b-94abbd4ad465
# pick a folder first

In [11]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

# update to be programmatic
tree = ET.parse("/data/CARDPB2/iNDI/Production/2b1c31ad-579d-4ea6-a782-251ea083ed6a/2b1c31ad-579d-4ea6-a782.xml")
OF_dir = Path("/data/CARDPB2/iNDI/Production/2b1c31ad-579d-4ea6-a782-251ea083ed6a/images")
index_tree = ET.parse('/data/CARDPB2/iNDI/Production/2b1c31ad-579d-4ea6-a782-251ea083ed6a/index/4e55b5d5-29d5-4260-bd9c.xml')

root = tree.getroot()

# Namespace mapping
ns = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'} # I believe this is consistent between all experiments

# Find elements
measurement_id = root.find('h:MeasurementID', ns).text
date = root.find('h:Date', ns).text
plate = root.find('h:InitialPlateName', ns).text

print("Measurement ID:", measurement_id)
print("Date:", date)
print("Plate:", plate)

root = index_tree.getroot()

# Find elements
plate_id = root.find('.//h:PlateID', ns).text
x_res = float(root.find('.//h:ImageResolutionX', ns).text) * 1e6
y_res = float(root.find('.//h:ImageResolutionY', ns).text) * 1e6

print("Plate:", plate_id)
print(f"X resolution: {x_res} µm")
print(f"Y resolution: {y_res} µm")

channels = []

# Find the Map element that contains channel info entries
for map_el in root.findall(".//h:Map", ns):
    # Check first entry if it has a ChannelName child
    first_entry = map_el.find("h:Entry", ns)
    if first_entry is not None and first_entry.find("h:ChannelName", ns) is not None:
        # Found the Map with channel entries
        for entry in map_el.findall("h:Entry", ns):
            ch_id = entry.attrib.get("ChannelID")
            ch_name = entry.find("h:ChannelName", ns).text
            ch_type = entry.find("h:ChannelType", ns).text if entry.find("h:ChannelType", ns) is not None else None
            excitation = entry.find("h:MainExcitationWavelength", ns).text if entry.find("h:MainExcitationWavelength", ns) is not None else None
            emission = entry.find("h:MainEmissionWavelength", ns).text if entry.find("h:MainEmissionWavelength", ns) is not None else None

            channels.append({
                "ChannelID": int(ch_id) if ch_id is not None else None,
                "Channel_name": ch_name,
                "Type": ch_type,
                "Excitation_nm": excitation,
                "Emission_nm": emission
            })
        break  

# Convert to DataFrame
channel_df = pd.DataFrame(channels).sort_values('ChannelID').reset_index(drop=True)
print(channel_df)

channel_df["Pseudocolor"] = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
channel_df["MPL_colormap"] = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
channel_df["Measurement_ID"] = measurement_id
channel_df["Measurement_date"] = date
channel_df["Plate_ID"] = plate_id
channel_df["res_x"] = x_res
channel_df["res_y"] = y_res

print(channel_df)

Measurement ID: 2b1c31ad-579d-4ea6-a782-251ea083ed6a
Date: 2026-02-12T20:12:48.0331159-05:00
Plate: INDI00002D
Plate: INDI00002D
X resolution: 0.09418670872212731 µm
Y resolution: 0.09418670872212731 µm
   ChannelID Channel_name          Type Excitation_nm Emission_nm
0          1         DAPI  Fluorescence           375         456
1          2    Alexa 647  Fluorescence           640         706
2          3    Alexa 488  Fluorescence           488         522
3          4    Alexa 568  Fluorescence           561         599
   ChannelID Channel_name          Type Excitation_nm Emission_nm Pseudocolor  \
0          1         DAPI  Fluorescence           375         456        blue   
1          2    Alexa 647  Fluorescence           640         706     magenta   
2          3    Alexa 488  Fluorescence           488         522       green   
3          4    Alexa 568  Fluorescence           561         599         red   

                                        MPL_colormap  \
0  <m

In [12]:
import re
import pandas as pd

# Collect all .tiff files
files = sorted([f for f in OF_dir.rglob('*') if f.suffix.lower() in ['.tiff']])

# Define a function to parse the filename
def parse_filename(name):
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    if match:
        return [int(g) for g in match.groups()]
    else:
        return [None] * 6  # Fallback if filename doesn't match

# Create a DataFrame with full path and relative subdirectory
df = pd.DataFrame({
    'filepath': files,
    'filename': [f.name for f in files], 
    'subdirectory': [f.parent.relative_to(OF_dir) for f in files]
})

# Extract metadata from filenames
df[['Row', 'Column', 'Frame', 'Plane', 'ChannelID', 'Time']] = df['filename'].apply(
    lambda x: pd.Series(parse_filename(x))
)

print(df.head())

                                            filepath  \
0  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
1  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
2  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
3  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
4  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   

                    filename subdirectory  Row  Column  Frame  Plane  \
0  r01c01f01p01-ch01t01.tiff       r01c01    1       1      1      1   
1  r01c01f01p01-ch02t01.tiff       r01c01    1       1      1      1   
2  r01c01f01p01-ch03t01.tiff       r01c01    1       1      1      1   
3  r01c01f01p01-ch04t01.tiff       r01c01    1       1      1      1   
4  r01c01f02p01-ch01t01.tiff       r01c01    1       1      2      1   

   ChannelID  Time  
0          1     1  
1          2     1  
2          3     1  
3          4     1  
4          1     1  


In [13]:
merged_ChannelID_df = pd.merge(df, channel_df, on="ChannelID")
print(merged_ChannelID_df)

                                                filepath  \
0      /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
1      /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
2      /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
3      /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
4      /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
...                                                  ...   
53755  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
53756  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
53757  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
53758  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   
53759  /data/CARDPB2/iNDI/Production/2b1c31ad-579d-4e...   

                        filename subdirectory  Row  Column  Frame  Plane  \
0      r01c01f01p01-ch01t01.tiff       r01c01    1       1      1      1   
1      r01c01f01p01-ch02t01.tiff       r01c01    1       1      1      1   
2      r01c01f01p01-ch03t01.tiff       r01c01    1 

In [14]:
summary = {
    "wells": merged_ChannelID_df[['Row', 'Column']].drop_duplicates().shape[0],
    "channels": merged_ChannelID_df['ChannelID'].nunique(),
    "z_planes": merged_ChannelID_df['Plane'].nunique(),
    "frames": merged_ChannelID_df['Frame'].nunique(),
    "timepoints": merged_ChannelID_df['Time'].nunique(),
}

print(summary)

print(f"""
Experiment ID: {measurement_id}
Plate ID: {plate}
Wells imaged: {summary["wells"]}
Frames per well: {summary["frames"]}
Channels per image: {summary["channels"]}
Z-slices per image: {summary["z_planes"]}
Timepoints per image: {summary["timepoints"]}
""")

{'wells': 384, 'channels': 4, 'z_planes': 1, 'frames': 35, 'timepoints': 1}

Experiment ID: 2b1c31ad-579d-4ea6-a782-251ea083ed6a
Plate ID: INDI00002D
Wells imaged: 384
Frames per well: 35
Channels per image: 4
Z-slices per image: 1
Timepoints per image: 1



In [15]:
import dask
from dask import delayed, compute
import pandas as pd
import tifffile
import numpy as np
from skimage import filters, morphology
from scipy import ndimage as ndi
from skimage.measure import regionprops_table
from scipy.stats import norm
import os

# --- Parameters ---
intensity_scaling_param = [1, 7]
blur_sigma = 1
min_area = 3500

# --- Function to process a single image ---
@delayed
def process_nucleus_image(image_path):
    # Load image
    nuc = tifffile.imread(image_path)
    
    # Normalize
    m, s = norm.fit(nuc.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, nuc.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, nuc.max())
    nuc_stretch = np.clip(nuc, stretch_min, stretch_max)
    image_norm = (nuc_stretch - stretch_min) / (stretch_max - stretch_min)

    # Blur
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # Step 1: Low level threshold
    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2
    img_low_level = blurred > th_low_cutoff
    img_low_level = morphology.remove_small_objects(img_low_level.astype(bool), min_size=min_area)
    img_low_level = morphology.dilation(img_low_level, footprint=morphology.disk(2))

    # Step 2: High level threshold
    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level)
    lab_low, num_obj = morphology.label(img_low_level, return_num=True)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        local_otsu = filters.threshold_otsu(blurred[single_obj])
        if local_otsu > otsu_cutoff:
            img_high_level[np.logical_and(blurred > 0.98 * local_otsu, single_obj)] = 1

    filled = ndi.binary_fill_holes(img_high_level)
    filled = morphology.dilation(filled, footprint=morphology.disk(2))
    labeled_filled = morphology.label(filled)
    nuc_seg = morphology.remove_small_objects(labeled_filled, min_size=min_area)

    # Extract features
    props = regionprops_table(
        label_image=nuc_seg,
        intensity_image=nuc,
        properties=(
            'label', 'area', 'mean_intensity', 'max_intensity', 'min_intensity', 'std_intensity',
            'centroid', 'eccentricity', 'solidity', 'perimeter', 'feret_diameter_max',
            'orientation', 'major_axis_length', 'minor_axis_length'
        )
    )
    df = pd.DataFrame(props)
    df['image_name'] = os.path.basename(image_path)
    
    return df

In [16]:
# Filter for DAPI images
dapi_samples = merged_ChannelID_df[merged_ChannelID_df['Channel_name'] == "DAPI"].copy() 

# Extract file paths
filepaths = dapi_samples['filepath'].tolist()

# --- Wrap all images with delayed ---
tasks = [process_nucleus_image(fp) for fp in filepaths]

In [17]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    dfs = dask.compute(*tasks, scheduler="processes")

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv("nuclei_features_test_20260317.csv", index=False)

[                                        ] | 0% Completed | 9.45 s ms

/tmp/ipykernel_1531044/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[                                        ] | 0% Completed | 25.18 s

/tmp/ipykernel_1531044/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[                                        ] | 0% Completed | 28.13 s

...

[##                                      ] | 7% Completed | 260.18 s

/tmp/ipykernel_1531044/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[##                                      ] | 7% Completed | 264.44 s


/tmp/ipykernel_1531044/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


KeyboardInterrupt: 